In [0]:
%run ../00_config/project_config

database,volume_name
raw,github_volume


In [0]:
silver_table = (
    f"{CATALOG}.{SILVER_SCHEMA}.github_events_silver"
)

silver_df = spark.table(
    silver_table
)

KPI 1 — Top Event Types

In [0]:
from pyspark.sql.functions import count

top_event_types = (
    silver_df
        .groupBy("event_type")
        .agg(
            count("*").alias("total_events")
        )
        .orderBy(
            "total_events",
            ascending=False
        )
)

In [0]:
display(top_event_types)

event_type,total_events
PushEvent,1615
CreateEvent,228
DeleteEvent,47


KPI 2 — Top Repositories

In [0]:
top_repositories = (
    silver_df
        .groupBy("repo_name")
        .agg(
            count("*").alias("total_events")
        )
        .orderBy(
            "total_events",
            ascending=False
        )
)

In [0]:
display(top_repositories)

repo_name,total_events
vSkeezy/test2,19
Fadil123-hah/DILZXJASHER,19
ghostventuresai/ghost_activepieces,15
bolividob/bibkreyol-comments-data,10
sukduqon/zyxpgh,10
themeganews101/meganews-RSS-Hub,9
Vecens/Signal-Permafrost,8
cloudwalker171/atlas-manifests,7
LabRatty/labratty.github.io,7
eternalreturn-git/portf,6


KPI 3 — Top GitHub Users

In [0]:
top_users = (
    silver_df
        .groupBy("actor_login")
        .agg(
            count("*").alias("total_events")
        )
        .orderBy(
            "total_events",
            ascending=False
        )
)

In [0]:
display(top_users)

actor_login,total_events
github-actions[bot],296
ghostventuresai,25
Fadil123-hah,19
vSkeezy,19
dependabot[bot],14
cursor[bot],10
pull[bot],10
sukduqon,10
bolividob,10
themeganews101,9


KPI 4 — Daily Activity

In [0]:
from pyspark.sql.functions import to_date

daily_activity = (
    silver_df
        .withColumn(
            "event_date",
            to_date("created_at")
        )
        .groupBy("event_date")
        .agg(
            count("*").alias("total_events")
        )
        .orderBy("event_date")
)

In [0]:
display(daily_activity)

event_date,total_events
2026-06-04,150
2026-06-07,750
2026-06-08,150
2026-06-09,300
2026-06-10,510
2026-06-13,30


In [0]:
top_event_types.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{CATALOG}.{GOLD_SCHEMA}.top_event_types"
    )

In [0]:
top_repositories.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{CATALOG}.{GOLD_SCHEMA}.top_repositories"
    )

In [0]:
top_users.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{CATALOG}.{GOLD_SCHEMA}.top_users"
    )

In [0]:
daily_activity.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{CATALOG}.{GOLD_SCHEMA}.daily_activity"
    )